In [1]:
# import the path tools and pndas
from pathlib import Path
import pandas as pd

# start with the folder where the notebook kernel is running 
project_root = Path.cwd()

# If the kernel started inside notebooks/, move up to the project root.
if project_root.name == "notebooks":
    project_root = project_root.parent

# build the path to the raw CSV
csv_path = project_root / "data" / "raw" / "BPI_Challenge_2019.csv"

#read the header and load 0 event rows
header = pd.read_csv(csv_path, nrows=0)

# column count and exact names in scource order
print("Column count: ", len(header.columns))
print("Column names: ", header.columns.tolist())

Column count:  22
Column names:  ['eventID ', 'case Spend area text', 'case Company', 'case Document Type', 'case Sub spend area text', 'case Purchasing Document', 'case Purch. Doc. Category name', 'case Vendor', 'case Item Type', 'case Item Category', 'case Spend classification text', 'case Source', 'case Name', 'case GR-Based Inv. Verif.', 'case Item', 'case concept:name', 'case Goods Receipt', 'event User', 'event org:resource', 'event concept:name', 'event Cumulative net worth (EUR)', 'event time:timestamp']


In [4]:
# List the exact source column names we want to inspect.
columns_to_preview = [
    "case concept:name",
    "case Purchasing Document",
    "case Item",
    "eventID ",  # The source header includes a trailing space.
    "event concept:name",
    "event time:timestamp",
    "case Item Category",
    "case GR-Based Inv. Verif.",
    "case Goods Receipt",
]

# Read the first 12 rows, keeping identifiers and timestamps as text.
preview = pd.read_csv(
    csv_path,
    usecols=columns_to_preview,
    nrows=12,
    dtype="string",
    keep_default_na=False,
)

# Show the columns in the order we listed them above.
display(preview[columns_to_preview])

,case concept:name,case Purchasing Document,case Item,eventID,event concept:name,event time:timestamp,case Item Category,case GR-Based Inv. Verif.,case Goods Receipt
0,4507004931_00020,4507004931,00020,65781719105536,Vendor creates invoice,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
1,4507004931_00010,4507004931,00010,65777424138241,Vendor creates debit memo,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
2,4507004931_00010,4507004931,00010,65777424138240,Vendor creates invoice,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
3,4507004931_00050,4507004931,00050,65794604007424,Vendor creates invoice,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
4,4507004931_00050,4507004931,00050,65794604007425,Vendor creates debit memo,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
5,4507004931_00040,4507004931,00040,65790309040129,Vendor creates debit memo,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
6,4507004931_00040,4507004931,00040,65790309040128,Vendor creates invoice,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
7,4507004931_00030,4507004931,00030,65786014072833,Vendor creates debit memo,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
8,4507004931_00030,4507004931,00030,65786014072832,Vendor creates invoice,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true
9,4507004931_00020,4507004931,00020,65781719105537,Vendor creates debit memo,26-01-1948 23:59:00.000,"3-way match, invoice before GR",false,true


In [5]:
# Import a dictionary-like tool that adds counts across chunks.
from collections import Counter

# Store the exact source column name and start with no counts.
category_column = "case Item Category"
category_counts = Counter()

# Read just the category column, up to 100,000 rows at a time.
chunks = pd.read_csv(
    csv_path,
    usecols=[category_column],
    dtype="string",
    keep_default_na=False,
    chunksize=100_000,
)

# Count the labels in each chunk and add them to the running totals.
for chunk in chunks:
    counts_in_chunk = chunk[category_column].value_counts()
    category_counts.update(counts_in_chunk.to_dict())

# Print the total event rows and each category from most to least frequent.
print("Total event rows:", sum(category_counts.values()))

for label, count in category_counts.most_common():
    print(f"{label!r}: {count:,}")

Total event rows: 1595923
'3-way match, invoice before GR': 1,234,708
'3-way match, invoice after GR': 319,233
'Consignment': 36,084
'2-way match': 5,898


In [6]:
# Use the exact column names from the CSV header.
case_column = "case concept:name"
document_column = "case Purchasing Document"
item_column = "case Item"
key_columns = [case_column, document_column, item_column]

# Start counters for the full-file check.
rows_checked = 0
rows_with_blank_keys = 0
rows_with_mismatched_case_ids = 0

# Read the three key columns in manageable chunks.
key_chunks = pd.read_csv(
    csv_path,
    usecols=key_columns,
    dtype="string",
    keep_default_na=False,
    chunksize=100_000,
)

for chunk in key_chunks:
    # Find rows where any key field is empty or contains only whitespace.
    blank_key = (
        chunk[case_column].str.strip().eq("")
        | chunk[document_column].str.strip().eq("")
        | chunk[item_column].str.strip().eq("")
    )

    # Compare case IDs only on rows that have all three fields.
    complete_keys = chunk.loc[~blank_key]
    expected_case_id = complete_keys[document_column] + "_" + complete_keys[item_column]
    mismatched_case_id = complete_keys[case_column].ne(expected_case_id)

    # Add this chunk's counts to the running totals.
    rows_checked += len(chunk)
    rows_with_blank_keys += blank_key.sum()
    rows_with_mismatched_case_ids += mismatched_case_id.sum()

# Print the results so we can reconcile them with the previous cell.
print(f"Event rows checked: {rows_checked:,}")
print(f"Rows with a blank key field: {rows_with_blank_keys:,}")
print(f"Rows with a mismatched case ID: {rows_with_mismatched_case_ids:,}")

Event rows checked: 1,595,923
Rows with a blank key field: 0
Rows with a mismatched case ID: 0
